In [ ]:
%%capture
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
from unsloth import FastLanguageModel 
import torch 
from google.colab import userdata


In [ ]:
# Import weights and biases
import wandb
run = wandb.init(
   project='Fine-Tuning-with-LoRA-using-unsloth-with-llama.3.1 8b-model',
   job_type="training",
   anonymous="allow",
   token=userdata.get('w&b')
)

wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: ERROR Invalid API key: API key may only contain the letters A-Z, digits and underscores.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

...

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nourannabil13 (N-team) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


In [ ]:
# Model Paramter Setup:
max_seq_length = 2048 
dtype = None
load_in_4bit = True 
model_name = "unsloth/Meta-Llama-3.1-8B-Instruct"

In [ ]:
# Load Model :

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit
)

==((====))==  Unsloth 2026.3.8: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


In [ ]:
# Start Fine-Tuning By Add LoRA To Model

model = FastLanguageModel.get_peft_model(
    model,
    r = 16, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16, 
    lora_dropout = 0, 
    bias = "none",    
    use_gradient_checkpointing="unsloth", 
    random_state = 3407,
    use_rslora = False,  
    loftq_config = None,
)

Unsloth 2026.3.8 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
# Load Dataset
from datasets import load_dataset
dataset = load_dataset("griffin/election-synthetic", split="train")

README.md:   0%|          | 0.00/459 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/162k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/849 [00:00<?, ? examples/s]

In [ ]:
dataset

Dataset({
    features: ['question', 'answer', 'meta', 'question_type', 'fact', 'rationale'],
    num_rows: 849
})

In [ ]:
print(dataset[:5])

{'question': ['Who did Kamala Harris select to be her vice presidential candidate in her campaign against Donald Trump?', 'True or False?\n\nAll governors are potential vice presidential candidates.', 'Who is leading the Democratic ticket in the 2024 presidential race?', 'If Kamala Harris and Tim Walz are standing 5 meters apart and a photographer wants to take their picture, what is the minimum focal length needed for a full-body shot if the camera sensor is 36mm wide and they occupy 80% of the frame width?', 'For which White House contender is the current leader of the Gopher State serving as the prospective vice president?'], 'answer': ["Tim Walz, Minnesota's Governor", 'False', 'Democratic presidential nominee', '4000mm', 'Kamala Harris'], 'meta': ['{"entity": "Tim Walz, Governor of Minnesota", "hops": 0}', '{"claim": "All governors are potential vice presidential candidates."}', '{"entity": "Presidential nominee", "hops": 0}', '{"reasoning_type": "physical"}', '{"hops": 4, "multi_

In [ ]:
print(dataset[:5]['question'])

['Who did Kamala Harris select to be her vice presidential candidate in her campaign against Donald Trump?', 'True or False?\n\nAll governors are potential vice presidential candidates.', 'Who is leading the Democratic ticket in the 2024 presidential race?', 'If Kamala Harris and Tim Walz are standing 5 meters apart and a photographer wants to take their picture, what is the minimum focal length needed for a full-body shot if the camera sensor is 36mm wide and they occupy 80% of the frame width?', 'For which White House contender is the current leader of the Gopher State serving as the prospective vice president?']


In [ ]:
print(dataset[:5]['answer'])

["Tim Walz, Minnesota's Governor", 'False', 'Democratic presidential nominee', '4000mm', 'Kamala Harris']


In [ ]:
dataset_sample = dataset.select(range(5))
display(dataset_sample.to_pandas())

,question,answer,meta,question_type,fact,rationale
0,Who did Kamala Harris select to be her vice pr...,"Tim Walz, Minnesota's Governor","{""entity"": ""Tim Walz, Governor of Minnesota"", ...",single_hop,"The governor of Minnesota, Tim Walz, was chose...",
1,True or False?\n\nAll governors are potential ...,False,"{""claim"": ""All governors are potential vice pr...",adversarial,"The governor of Minnesota, Tim Walz, was chose...","While Tim Walz, a governor, was chosen as a vi..."
2,Who is leading the Democratic ticket in the 20...,Democratic presidential nominee,"{""entity"": ""Presidential nominee"", ""hops"": 0}",single_hop,"The governor of Minnesota, Tim Walz, was chose...",
3,If Kamala Harris and Tim Walz are standing 5 m...,4000mm,"{""reasoning_type"": ""physical""}",reasoning,"The governor of Minnesota, Tim Walz, was chose...",Kamala Harris and Tim Walz are running mates i...
4,For which White House contender is the current...,Kamala Harris,"{""hops"": 4, ""multi_hop_path"": [""Tim Walz"", ""is...",multi_hop,"The governor of Minnesota, Tim Walz, was chose...","Tim Walz is the governor of Minnesota, also kn..."


In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1"
)

In [ ]:
def formatting_prompts_func(examples):
    texts = []
    convos = []  # Create a list to store conversations

    for question, answer in zip(examples['question'], examples['answer']):
        # Create a conversation-like structure
        convo = [
            {'role': 'user', 'content': question},
            {'role': 'assistant', 'content': answer}
        ]
        convos.append(convo)  # Add convo to the list

        # Apply the chat template
        formatted_text = tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        )
        texts.append(formatted_text)

    return {'text': texts, 'conversations': convos}  # Return the list of convos
pass

In [ ]:
dataset = dataset.map(formatting_prompts_func, batched = True)

Map:   0%|          | 0/849 [00:00<?, ? examples/s]

In [ ]:
dataset 

Dataset({
    features: ['question', 'answer', 'meta', 'question_type', 'fact', 'rationale', 'text', 'conversations'],
    num_rows: 849
})

In [ ]:
dataset[5]["conversations"]

[{'content': 'Should Tim Walz be morally obligated to resign as governor of Minnesota upon accepting the vice presidential nomination?',
  'role': 'user'},
 {'content': "While there's no legal requirement, there is a strong moral argument that Tim Walz should resign as governor of Minnesota upon accepting the vice presidential nomination. This is because:\n\n1. Full commitment: The roles of governor and vice presidential candidate both demand full-time attention. Resigning would allow Walz to fully commit to the campaign without shortchanging his gubernatorial duties.\n\n2. Avoiding conflicts of interest: As a sitting governor, Walz might face situations where his decisions could be perceived as motivated by campaign interests rather than the best interests of Minnesota.\n\n3. Fair representation for Minnesota: The state deserves a full-time governor focused solely on state issues, not divided attention.\n\n4. Transparency and integrity: Resigning would demonstrate a commitment to ethi

In [ ]:
dataset[5]["text"]

"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nShould Tim Walz be morally obligated to resign as governor of Minnesota upon accepting the vice presidential nomination?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nWhile there's no legal requirement, there is a strong moral argument that Tim Walz should resign as governor of Minnesota upon accepting the vice presidential nomination. This is because:\n\n1. Full commitment: The roles of governor and vice presidential candidate both demand full-time attention. Resigning would allow Walz to fully commit to the campaign without shortchanging his gubernatorial duties.\n\n2. Avoiding conflicts of interest: As a sitting governor, Walz might face situations where his decisions could be perceived as motivated by campaign interests rather than the best interests of Minnesota.\n\n3. Fair representatio

In [ ]:
# Train The Model

from trl import SFTTrainer  # Supervised Fine Tuning Trainer 
from transformers import TrainingArguments , DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text", 
    max_seq_length = max_seq_length, 
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer), 
    dataset_num_proc = 2, 
    packing = False, 
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(), 
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/849 [00:00<?, ? examples/s]

In [ ]:
from unsloth import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

Map (num_proc=6):   0%|          | 0/849 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/849 [00:00<?, ? examples/s]

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 849 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
1,5.176304
2,3.586127
3,3.194402
4,3.889051
5,3.185266
6,1.882103
7,2.367249
8,2.215227
9,1.577129
10,1.218171


In [ ]:
# Save the fine-tuned model
wandb.finish()

In [ ]:
# inference
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
        (layers): ModuleList(
          (0): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [ ]:
# The Messages Example

# One Question
messages = [
    {"role": "user", "content": "who is partnering with Kamala Harris on the Democratic ticket In the 2024 presidential race?"},
]

# Full Converstion
messages1 = [
 {"role":"user","content":"Hello"},
 {"role":"assistant","content":"Hi!"},
 {"role":"user","content":"Who won the election?"}
]

In [ ]:

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, 
    return_tensors = "pt", 
).to("cuda") 



In [ ]:
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)

In [ ]:
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = False, temperature = 1.5, min_p = 0.1)

Tim<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

who is partnering with Kamala Harris on the Democratic ticket In the 2024 presidential race?<|eot_id|><|start_header_id|>assistant<|end_header_id|>



Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Tim Walz<|eot_id|>


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
# Saving, loading finetuned models
model.save_pretrained("Meta-Llama-3.1-8B-Instruct/Questions") # Local saving
tokenizer.save_pretrained("Meta-Llama-3.1-8B-Instruct/Questions")

('Meta-Llama-3.1-8B-Instruct/Questions/tokenizer_config.json',
 'Meta-Llama-3.1-8B-Instruct/Questions/chat_template.jinja',
 'Meta-Llama-3.1-8B-Instruct/Questions/tokenizer.json')

To Save the model in Hugging Face

In [ ]:
model.push_to_hub("nourannabil/Meta-Llama-3.1-8B-Instruct-Questions" ,
                  token = userdata.get("HF_ACCESS_TOKEN")) # Online saving


README.md:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  57%|#####7    | 95.9MB /  168MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Saved model to https://huggingface.co/nourannabil/Meta-Llama-3.1-8B-Instruct-Questions


# Merge model with lora weights and save to gguf
You can then do inference locally with Ollama or llama.cpp

Popular quantization methods
* q4_k_m
4bit quantization. Low memory. All models you pull with ollama uses this quantization.
* q8_0
8bit quantization. Medium memory.
* f16
16 bit quantization. A lot of models are already in 16 bit so then no quantization happens

* not_quantized
Often same as f16.

In [ ]:
model.push_to_hub_gguf(
    "Meta-Llama-3.1-8B-q4_k_m-paul-Questions-GGUF",
    tokenizer,
    quantization_method = "q4_k_m",
    token = userdata.get('HF_ACCESS_TOKEN')
  )

Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...1-8b-instruct.Q4_K_M.gguf:   0%|          |  559kB / 4.92GB            

Uploading config.json...
Uploading Ollama Modelfile...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/nourannabil/Meta-Llama-3.1-8B-q4_k_m-paul-Questions-GGUF
Unsloth: Cleaning up temporary files...


'nourannabil/Meta-Llama-3.1-8B-q4_k_m-paul-Questions-GGUF'

### Load model and saved lora adapters

For if you want to continue finetuning or want to do inference using the model in safetensor format.

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "nourannabil/Meta-Llama-3.1-8B-Instruct-Questions",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
    token=userdata.get('HF_ACCESS_TOKEN')
)

FastLanguageModel.for_inference(model)

messages = [
    {"role": "user", "content": "who is partnering with Kamala Harris on the Democratic ticket In the 2024 presidential race?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)


_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = False, temperature = 1.5, min_p = 0.1)
